In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import sys
import pickle
import random
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import os
import sys
import torch
import numpy as np
import torch.nn as nn
import logging
logging.basicConfig(level=logging.INFO)
from pathlib import Path
import torchvision.datasets as Datasets
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai import transforms
from torch.utils.data import Dataset
from torch.nn.utils import clip_grad_norm_
from torch.optim import Adam
import random
from skimage.transform import rotate
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics import MeanSquaredError, R2Score 
from skimage import measure 
from torch.utils.data import DataLoader
import torch.nn.init as init

In [32]:
pkg_path = str(Path(os.getcwd()).parent.absolute())  # Get parent directory of the current working directory
sys.path.insert(0, pkg_path)

# If you need to include a 'src' directory relative to the current notebook:
src_path = os.path.abspath(os.path.join(os.getcwd(), '../src'))  # Adjust path relative to the current working directory
sys.path.insert(0, src_path)
print(f"Path to the 'src' directory: {src_path}")
data_path=pkg_path+'/Datasets/UPENN_GBM/csvs/'
# Now you can import your modules from the 'src' directory
# from src import *

Path to the 'src' directory: /home/ee577/project/src


In [33]:
class UNet3DRegression_survival(nn.Module):
    def __init__(self, in_channels, out_channels, feature_importances=None, l1_lambda=1e-11,freeze_unet=False, device='cuda'):
        super(UNet3DRegression_survival, self).__init__()

        self.device = device  # Store device info
        
        # Initialize UNet with 3D structure for segmentation task
        self.unet = UNet(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,  # out_channels for segmentation (e.g., 1 for binary segmentation)
            channels=(16, 32, 64, 128),
            strides=(2, 2, 2),
            kernel_size=3,
            up_kernel_size=3,
            num_res_units=2,
            act='PReLU',
            norm='INSTANCE',
        ).to(self.device)  # Move the UNet model to the specified device
        self.freeze_unet = freeze_unet
        # Initialize dropout and move it to device
        self.dropout_UNET = nn.Dropout3d(p=0.3).to(self.device)
        self.dropout = nn.Dropout2d(p=0.3).to(self.device)
        self.segmentation_head = nn.Conv3d(out_channels, 1, kernel_size=1).to(self.device)  # Move segmentation head to device
        if feature_importances is not None:
            self.feature_importances=feature_importances
        self.l1_lambda = l1_lambda
        # Fully connected layers for regression (survival prediction)
        self.fc1 = nn.Identity().to(self.device)
        # Initializing the regression layer as None
        self.regression_layer = nn.Identity().to(self.device)

         # Fully connected layers for regression (survival prediction)
        self.fc1=nn.Identity().to(self.device)
        self.first_size=32
        self.fc2 = nn.Linear(self.first_size, 1).to(self.device) 
        self.freeze_layers(freeze_unet)

    def freeze_layers(self, freeze_unet):
        """
        Freezes or unfreezes the UNet and regression layers based on the flag `freeze_unet`.
        """
        for param in self.unet.parameters():
            param.requires_grad = not freeze_unet  

        if not isinstance(self.regression_layer, nn.Identity):
            for param in self.regression_layer.parameters():
                param.requires_grad = not freeze_unet 

    def forward(self, x):
        if x.ndimension() == 4:  # Shape: (channels, depth, height, width)
            x = x.unsqueeze(0).to(self.device)  # Move input to device

        _, _, depth, height, width = x.size()
        if depth % 16 != 0 or height % 16 != 0 or width % 16 != 0:
            new_depth = (depth // 16 + 1) * 16
            new_height = (height // 16 + 1) * 16
            new_width = (width // 16 + 1) * 16
            x = F.interpolate(x, size=(new_depth, new_height, new_width), mode='trilinear', align_corners=True)

        x = self.unet(x)  # Forward pass through UNet
        x = self.dropout_UNET(x)  # Apply dropout
        segmentation_output = self.segmentation_head(x)  # Get segmentation output
        segmentation_output = segmentation_output.to(self.device)  # Ensure output is on the correct device
        x = F.adaptive_avg_pool3d(x, (1, 1, 1))  # Global average pooling
        x = x.view(x.size(0), -1)  

        # Initialize regression_layer with the correct output size dynamically during the forward pass
        if isinstance(self.regression_layer, nn.Identity):
            self.regression_layer = nn.Linear(x.size(1), 55).to(self.device) 
        x = self.regression_layer(x)  # Apply regression layer
        self.dropout
        self.fc1 = nn.Linear(x.size(1), self.first_size).to(self.device)
        x = F.relu(self.fc1(x)) 
        self.dropout
        x = F.relu(self.fc2(x)) 
        
        survival_output = x
        return  survival_output


In [34]:
class UNet3DRegression(nn.Module):
    def __init__(self, in_channels, out_channels, segmentation_weight=0.8, l1_lambda=1e-11, device=device):
        super(UNet3DRegression, self).__init__()

        self.device = device  # Set the device for the model
        # Initialize UNet with 3D structure
        self.unet = UNet(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            channels=(16, 32, 64, 128),
            strides=(2, 2, 2),
            kernel_size=3,
            up_kernel_size=3,
            num_res_units=2,
            act='PReLU',
            norm='INSTANCE',
            dropout=0.1,
        ).to(device)  # Move UNet to the device

        # Initialize regression layer as None initially
        self.regression_layer = nn.Identity().to(device)
        

        # Initialize dropout and move it to device
        self.dropout = nn.Dropout3d(p=0.3).to(device)

        # Initialize segmentation weight and move it to device
        self.segmentation_weight = torch.tensor(segmentation_weight).to(device)  # Ensure segmentation weight is a tensor
        self.segmentation_head = nn.Conv3d(out_channels, 1, kernel_size=1).to(device)  # Move segmentation head to device

        self.l1_lambda = l1_lambda  # L1 regularization lambda

    def forward(self, x):
        if x.ndimension() == 4:  # Shape: (channels, depth, height, width)
            x = x.unsqueeze(0).to(self.device)  # Move input to device

        _, _, depth, height, width = x.size()
        if depth % 16 != 0 or height % 16 != 0 or width % 16 != 0:
            new_depth = (depth // 16 + 1) * 16
            new_height = (height // 16 + 1) * 16
            new_width = (width // 16 + 1) * 16
            x = F.interpolate(x, size=(new_depth, new_height, new_width), mode='trilinear', align_corners=True)

        x = self.unet(x)  # Forward pass through UNet
        x = self.dropout(x)  # Apply dropout
        segmentation_output = self.segmentation_head(x)  # Get segmentation output
        segmentation_output = segmentation_output.to(self.device)  # Ensure output is on the correct device

        # Apply global average pooling to reduce the spatial dimensions
        x = F.adaptive_avg_pool3d(x, (1, 1, 1))  # Global average pooling
        x = x.view(x.size(0), -1)  # Flatten to [batch_size, channels]

        # Initialize regression_layer with the correct output size dynamically during the forward pass
        if isinstance(self.regression_layer, nn.Identity):
            self.regression_layer = nn.Linear(x.size(1), 55).to(self.device)  # Adjust output to 55 and move to device

        regression_output = self.regression_layer(x)  # Apply regression layer (55 output features)
        regression_output = regression_output.to(self.device)  # Ensure output is on the correct device

        return  regression_output


In [35]:
def normalize_survival(images, survival):
    Z_log=np.log(survival)
    Z_log_norm = (Z_log - np.mean(Z_log)) / np.std(Z_log)
    t_half=np.mean(Z_log)
    lambda_value=np.log(2)/t_half
    Z_norm=np.exp(-Z_log_norm*lambda_value)
    Z_norm=(Z_norm-np.mean(Z_norm))
    valid_indices = abs(Z_norm) <= 0.3
    Z_norm = Z_norm[valid_indices]
    images = images[valid_indices]
    scaler = MinMaxScaler()
    Z_scaled = scaler.fit_transform(Z_norm.reshape(-1, 1))
    return images, Z_scaled, lambda_value, scaler

In [36]:
class CustomDataset(Dataset):
    def __init__(self, images, labels, masks=None, transform=None):
        """
        Args:
            images (numpy array or torch tensor): 4D tensor with shape (N, D, H, W), where N is the number of samples
            labels (numpy array or torch tensor): 2D tensor with shape (N, num_features), where N is the number of samples
            masks (numpy array or torch tensor, optional): 4D tensor with shape (N, D, H, W), where N is the number of samples
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.images = images
        self.labels = labels
        self.masks = masks  # Optional masks for segmentation
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Get the segmentation mask for this sample if provided
        if self.masks is not None:
            mask = self.masks[idx]
        else:
            mask = None

        # Convert to torch tensor if necessary
        if isinstance(image, np.ndarray):
            image = torch.tensor(image, dtype=torch.float32).to(device)

        label = torch.tensor(label, dtype=torch.float32).to(device)

        # If the image is 3D (D, H, W), add channel dimension (1, D, H, W)
        if image.ndimension() == 3:  # (D, H, W)
            image = image.unsqueeze(0)  # Add channel dimension (Shape becomes: (1, D, H, W))

        # Apply transformation to the image if specified
        if self.transform:
            image = self.transform(image)

        # Return the image, label, and mask (if available)
        if mask is not None:
            if isinstance(mask, np.ndarray):
                mask = torch.tensor(mask, dtype=torch.float32).to(device)
                if mask.ndimension() == 3:  # (D, H, W)
                    mask = mask.unsqueeze(0)
                return {'image': image, 'label': label, 'mask': mask}
        else:
            return {'image': image, 'label': label}

In [37]:
direct_pairs2='/home/ee577/project/training_data/DTI_AD_NC_merged_data.pkl'
direct_pairs1='/home/ee577/project/training_data/DSC_ED_merged_data.pkl'

with open(direct_pairs1, 'rb') as f:
    X, y, Z = pickle.load(f)  # X: images, y: features, Z: survival

# Normalize the survival data for the second pair
X_1_s, y_1_s ,lambda_value,scaler= normalize_survival(X, Z)  # Normalizing for the survival pair (X and Z)

# Split 1: Using X as input and y (features) as the target
X_train_1_f, X_temp_1_f, y_train_1_f, y_temp_1_f = train_test_split(
    X, y, test_size=0.2, random_state=22
)
X_val_1_f, X_test_1_f, y_val_1_f, y_test_1_f = train_test_split(
    X_temp_1_f, y_temp_1_f, test_size=0.7, random_state=22
)

# Split 2: Using X as input and Z (survival) as the target
X_train_1_s, X_temp_1_s, y_train_1_s, y_temp_1_s = train_test_split(
    X_1_s, y_1_s, test_size=0.2, random_state=22
)
X_val_1_s, X_test_1_s, y_val_1_s, y_test_1_s = train_test_split(
    X_temp_1_s, y_temp_1_s, test_size=0.7, random_state=22
)

with open(direct_pairs2, 'rb') as f:
    X, y, Z = pickle.load(f)  # X: images, y: features, Z: survival

# Normalize the survival data for the second pair
X_2_s, y_2_s,_,_ = normalize_survival(X, Z)  # Normalizing for the survival pair (X and Z)

# Split 1: Using X as input and y (features) as the target
X_train_2_f, X_temp_2_f, y_train_2_f, y_temp_2_f = train_test_split(
    X, y, test_size=0.2, random_state=22
)
X_val_2_f, X_test_2_f, y_val_2_f, y_test_2_f = train_test_split(
    X_temp_2_f, y_temp_2_f, test_size=0.7, random_state=22
)

# Split 2: Using X as input and Z (survival) as the target
X_train_2_s, X_temp_2_s, y_train_2_s, y_temp_2_s = train_test_split(
    X_2_s, y_2_s, test_size=0.2, random_state=22
)
X_val_2_s, X_test_2_s, y_val_2_s, y_test_2_s = train_test_split(
    X_temp_2_s, y_temp_2_s, test_size=0.7, random_state=22
)


In [38]:
batch_siz=16
test_loader_1_f = DataLoader(CustomDataset(X_test_1_f, y_test_1_f), batch_size=batch_siz, shuffle=False)

# For label 1_s (X -> Z)
test_loader_1_s = DataLoader(CustomDataset(X_test_1_s, y_test_1_s), batch_size=batch_siz, shuffle=False)

# For label 2_f (X_2 -> y_2)
test_loader_2_f = DataLoader(CustomDataset(X_test_2_f, y_test_2_f), batch_size=batch_siz, shuffle=False)

# For label 2_s (X_2 -> Z_2)
test_loader_2_s = DataLoader(CustomDataset(X_test_2_s, y_test_2_s), batch_size=batch_siz, shuffle=False)

for batch in test_loader_1_s:
    survival = batch['image']
    features = batch['label']
    print(f" image size: {survival.shape}, label size {features.shape}")

n_shape = batch['image'].shape 
out_shape=y_train_1_s.shape[1:][0]

 image size: torch.Size([16, 1, 74, 98, 86]), label size torch.Size([16, 1])
 image size: torch.Size([16, 1, 74, 98, 86]), label size torch.Size([16, 1])
 image size: torch.Size([16, 1, 74, 98, 86]), label size torch.Size([16, 1])
 image size: torch.Size([15, 1, 74, 98, 86]), label size torch.Size([15, 1])


In [39]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.autograd import Variable
from torchvision import transforms
from torch.utils.data import DataLoader

def generate_contour_saliency_map(model, input_image, target_label, loss_fn, device='cuda'):
    """
    Generates a contour-based saliency map for a given model and input.
    
    Args:
        model: The trained model (e.g., UNet3DRegression or UNet3DRegression_survival).
        input_image (torch.Tensor): The input image tensor (shape: [batch_size, channels, depth, height, width]).
        target_label (torch.Tensor): The target label tensor for the output of interest (e.g., survival output).
        loss_fn (callable): A custom loss function to use for gradient computation.
        device (str): The device to run the model on, default is 'cuda'.
        
    Returns:
        tuple: The original input image and the saliency map with contour plot.
    """
    
    model.eval()  # Set the model to evaluation mode
    input_image = input_image.to(device)  # Move the input image to the correct device
    target_label = target_label.to(device)  # Move target to device
    
    input_image.requires_grad = True  # Allow gradients to be computed with respect to the input
    
    # Forward pass
    output = model(input_image)  # Perform forward pass
    
    # Choose the output you are interested in (e.g., regression_output or survival_output)
    output_target = output # Modify based on your model's output structure
    
    # Calculate the custom loss for the chosen output
    loss = loss_fn(output_target, target_label)
    
    # Backward pass to calculate gradients with respect to input
    model.zero_grad()
    loss.backward()
    
    # Get the gradient of the loss with respect to the input image
    gradients = input_image.grad[0]  # Assuming batch size of 1 for simplicity
    
    # Calculate the absolute value of gradients
    saliency_map = gradients.abs().cpu().numpy()
    
    if len(saliency_map.shape) == 3:  # 3D case: (depth, height, width)
        # Take a horizontal slice (middle of the height axis)
        middle_height = saliency_map.shape[1] // 2
        saliency_map_slice = saliency_map[:, middle_height, :]  # Take middle horizontal slice (depth x width)

    elif len(saliency_map.shape) == 2:  # 2D case: (height, width)
        # If the saliency map is 2D, we just need to use the whole map
        saliency_map_slice = saliency_map
    else:
        raise ValueError(f"Unexpected saliency map shape: {saliency_map.shape}")

    # Return both the original image and the saliency map slice
    return input_image.cpu().detach().numpy(), saliency_map_slice

    # Convert to numpy for visualization
    input_image = input_image.cpu().detach().numpy()
    
    # Return both the original image and the saliency map
    return input_image, saliency_map


# Function to process a specific model and generate saliency maps

In [40]:
def load_model(model, feature_model_path, linear_model_path=None):
    state_dict = torch.load(feature_model_path)
    model.load_state_dict(state_dict, strict=False)
    if linear_model_path:
        linear_state_dict = torch.load(linear_model_path)
        model_state_dict = model.state_dict()
        linear_layer_keys = [key for key in linear_state_dict.keys() if 'fc' in key]  
        for key in linear_layer_keys:
            if key in model_state_dict:
                model_state_dict[key] = linear_state_dict[key]
        model.load_state_dict(model_state_dict)
    model.eval() 
    return model

In [41]:
model_DSC_feat='/home/ee577/project/best_models_unet/DSC_feat_best_model.pth'
model_DTI_feat='/home/ee577/project/best_models_unet/DTI_feat_best_model.pth'
model_DSC_surv='/home/ee577/project/best_models_unet/DSC_surv_best_model.pth'
model_DTI_surv='/home/ee577/project/best_models_unet/DTI_surv_best_model.pth'

feat_model=UNet3DRegression(in_channels=1, out_channels=out_shape, device=device)
model_DSC_feat=load_model(feat_model, model_DSC_feat).to(device)
model_DTI_feat=load_model(feat_model, model_DTI_feat).to(device)

surv_model=UNet3DRegression_survival(in_channels=1, out_channels=out_shape, device=device)
model_DSC_surv=load_model(surv_model, model_DSC_surv).to(device)
model_DTI_surv=load_model(surv_model, model_DTI_surv).to(device)

models = [model_DSC_feat, model_DTI_feat, model_DSC_surv, model_DTI_surv]
loss_fn = torch.nn.MSELoss()

def process_model_and_generate_saliency(model, test_loader, loss_fn, model_name, input_label_type):
    """
    Process the model, input the test_loader, and generate saliency maps.
    This function generates saliency maps for each model and saves the results.
    
    Args:
        model: The trained model to generate saliency maps for.
        test_loader: DataLoader with the test data.
        loss_fn: Loss function used for the gradient computation.
        model_name: Name of the model (used for file naming).
        input_label_type: Type of label (either 'f' for features or 's' for survival).
    """
    # Iterate over the test loader and generate saliency maps for the chosen image
    saliency_maps = {}
    for batch_idx, batch in enumerate(test_loader):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Generate saliency maps
        input_image, saliency_map = generate_contour_saliency_map(model, images[0], labels[0], loss_fn, device)
        
        # Store results in a dictionary with model name and label type
        saliency_maps[f'{model_name}_{input_label_type}_saliency_map_batch_{batch_idx}'] = {
            'image': input_image,
            'saliency_map': saliency_map
        }
    
    return saliency_maps

loss_fn = nn.MSELoss()  
models = {
    '1': {'DSC': model_DSC_feat, 'DTI': model_DTI_feat},
    '2': {'DSC': model_DSC_surv, 'DTI': model_DTI_surv}
}

label_types = ['f', 's']  # Feature and Survival

all_saliency_maps = {}

for model_key in models.keys():
    for label_type in label_types:
        # Select the corresponding model
        model = models[model_key]['DSC'] if model_key == '1' and label_type == 'f' else models[model_key]['DTI']
        
        # Select the corresponding DataLoader
        test_loader = globals()[f'test_loader_{model_key}_{label_type}']
        
        # Generate saliency maps
        saliency_maps = process_model_and_generate_saliency(
            model, test_loader, loss_fn, model_key, label_type
        )
        
        # Save results
        all_saliency_maps.update(saliency_maps)
        print(f"Processed saliency maps for {model_key}_{label_type}")




loss_fn = nn.MSELoss()  

models = {
    '1': {'DSC': model_DSC_feat, 'DTI': model_DTI_feat},
    '2': {'DSC': model_DSC_surv, 'DTI': model_DTI_surv}
}

label_types = ['f', 's']  # Feature and Survival

all_saliency_maps = {}

for model_key in models.keys():
    for label_type in label_types:
        # Select the corresponding model
        model = models[model_key]['DSC'] if model_key == '1' and label_type == 'f' else models[model_key]['DTI']
        
        # Select the corresponding DataLoader
        test_loader = globals()[f'test_loader_{model_key}_{label_type}']
        
        # Generate saliency maps
        saliency_maps = process_model_and_generate_saliency(
            model, test_loader, loss_fn, model_key, label_type
        )
        
        # Save results
        all_saliency_maps.update(saliency_maps)
        print(f"Processed saliency maps for {model_key}_{label_type}")
first_batch_saliency_maps = extract_first_batch_for_plotting(all_saliency_maps)
plot_saliency_maps_with_original(first_batch_saliency_maps, all_saliency_maps)


import pickle
saliency_maps_file = 'saliency_maps.pkl'
with open(saliency_maps_file, 'wb') as f:
    pickle.dump(all_saliency_maps, f)
print(f"Saliency maps saved to {saliency_maps_file}")


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import explained_variance_score
from sklearn.preprocessing import MinMaxScaler

# Loss function
loss_fn = nn.MSELoss()

# Function to inverse transform survival values
def inverse_transform_survival(Z_scaled, lambda_value, scaler):
    Z_norm = scaler.inverse_transform(Z_scaled.reshape(-1, 1)).flatten()
    Z_log_norm = -np.log(Z_norm) / lambda_value
    Z_log = Z_log_norm * np.std(Z_log_norm) + np.mean(Z_log_norm)
    Z = np.exp(Z_log)
    return Z

# Function to compute MAPE
def compute_mape(y_true, y_pred):
    epsilon = 1e-10  # Avoid division by zero
    return torch.mean(torch.abs((y_true - y_pred) / (y_true + epsilon))) * 100

# Function to compute Explained Variance
def compute_explained_variance(y_true, y_pred):
    return explained_variance_score(y_true.cpu().numpy(), y_pred.cpu().numpy())

# Models and DataLoaders
models = {
    '1': {'DSC': model_DSC_feat, 'DTI': model_DTI_feat},
    '2': {'DSC': model_DSC_surv, 'DTI': model_DTI_surv}
}

label_types = ['f', 's']  # Feature and Survival

# Dictionary to store all metrics for each sample
all_metrics = {}

# Iterate over all combinations of model types and label types
for model_key in models.keys():
    for label_type in label_types:
        # Select the corresponding model
        model = models[model_key]['DSC'] if model_key == '1' and label_type == 'f' else models[model_key]['DTI']
        
        # Select the corresponding DataLoader
        test_loader = globals()[f'test_loader_{model_key}_{label_type}']
        
        # Initialize list to store metrics for each batch
        sample_metrics = []
        
        # Set model to evaluation mode
        model.eval()

        # Iterate over the test batches
        with torch.no_grad():
            for batch in test_loader:
                survival = batch['image']
                features = batch['label']
                predictions = model(inputs)
                
                # Compute MSE
                MSE = loss_fn(predictions, labels)

                # Compute MAPE
                mape = compute_mape(labels, predictions)
                
                # Inverse transform predictions and labels to original scale
                rescaled_survival_pred = inverse_transform_survival(predictions.cpu().numpy(), lambda_value, scaler)
                rescaled_survival_labels = inverse_transform_survival(labels.cpu().numpy(), lambda_value, scaler)

                # Compute Explained Variance
                explained_var = compute_explained_variance(rescaled_survival_labels, rescaled_survival_pred)
                
                # Compute RMSE
                rmse = torch.sqrt(MSE)

                # Store metrics for each sample
                for i in range(len(labels)):
                    sample_metrics.append({
                        'predicted_surv': rescaled_survival_pred[i],
                        'actual_surv': rescaled_survival_labels[i],
                        'mse': MSE.item(),
                        'mape': mape.item(),
                        'explained_variance': explained_var,
                        'rmse': rmse.item()
                    })

        # Save the metrics for this model and label type
        all_metrics[f"{label_type}_{model_key}_DSC" if model_key == '1' else f"{label_type}_{model_key}_DTI"] = sample_metrics
        print(f"Processed metrics for {label_type}_{model_key}_DSC" if model_key == '1' else f"Processed metrics for {label_type}_{model_key}_DTI")


AttributeError: 'str' object has no attribute 'ndimension'

In [ ]:
# Example: Access the metrics for the first model
first_model_metrics = all_metrics['1_f']  # Metrics for DSC model with 'f' features

# Print the first sample's metrics (just an example)
print(first_model_metrics[0])
